# Team pressing structure -- HOW each team presses (with automatic real clips)

Built on Step 1 (frame tables) and Step 2 (pressure events / press sequences). Everything is computed from the data:
no names, no manual labels -- positions and teams come from the tracking and the kits.

1. **Where**: every press sequence placed on a 3 x 3 grid -- own / middle / attacking third x left / centre / right,
   always seen from the *pressing* team (attacking left->right); counts + regain rate per cell
2. **Inferred positions** (from average positions in possession, no names): GK / DEF / MID / FWD and left / centre /
   right -> "on whom" a team presses
3. **Situations**: opponent possessions start as *goal kick*, *goalkeeper in open play*, *other restart* or *open play*;
   each press is classified: **on the goalkeeper, on goal kicks, on defenders, on midfielders, on forwards, counter-press**
4. **Marking**: man-oriented (1 v 1, following a fixed opponent) vs zonal -- per player and per team, + the main 1 v 1
   match-ups in the high press
5. **Pressing shape**: average out-of-possession shape and formation (e.g. 4-4-2 / 4-2-4) in **high / mid / low block**,
   width, depth, distance between lines
6. **Real clips**, chosen automatically (best example per team x situation): the match video with the pressing team's
   shape, pressers -> carrier lines, a 5 m ring on the carrier, a banner (situation, pressers, outcome) and a mini-pitch
7. a JSON report per team (for the LLM step)

In [ ]:
import os, json
import numpy as np
import pandas as pd
import cv2
import pickle, io
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from collections import Counter, defaultdict

CACHE_DIR = r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache"
VIDEO_PATH = r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\clip.mkv"
PLAYER_TABLE = os.path.join(CACHE_DIR, "player_frame_table_named.parquet")
BALL_TABLE   = os.path.join(CACHE_DIR, "ball_frame_table_named.parquet")
META_PATH    = os.path.join(CACHE_DIR, "match_meta_named.json")
PRESS_DIR    = os.path.join(CACHE_DIR, "pressure")                       # Step 2 outputs
HOMOGRAPHY_PATH = os.path.join(CACHE_DIR, "homography_cache_barca_atletico_firsthalf.pkl")
OUT_DIR      = os.path.join(PRESS_DIR, "team_structure")
CLIP_DIR     = os.path.join(OUT_DIR, "clips")

TEAM_COLORS = {"BAR": (152, 77, 0), "ATM": (36, 53, 203)}               # BGR, for clips / figures (by team name)
GK_RESTART_GAP_S = 3.0          # no carrier for >= 3 s before the goalkeeper gets it in his box = goal kick
GOAL_KICK_PHASE_S = 10.0        # presses within 10 s of a goal kick count as "pressing the goal kick"
MARK_RADIUS_M = 8.0             # a defender "marks" his nearest opponent if within 8 m
MARK_SAMPLE_EVERY = 5           # frames
SHAPE_MIN_VISIBLE = 8           # a frame counts for the team shape only if >= 8 outfield players of the team are visible
LINE_GAP_M = 5.0                # two players are in different lines if their average depths differ by > 5 m
CLIPS_PER_SITUATION = 2
CLIP_PRE_S, CLIP_POST_S = 2.0, 3.0
RENDER_CLIPS = True
PX_PER_METER = 10.0

os.makedirs(CLIP_DIR, exist_ok=True)
pf = pd.read_parquet(PLAYER_TABLE)
bt = pd.read_parquet(BALL_TABLE).set_index("frame_idx").sort_index()
meta = json.load(open(META_PATH, encoding="utf-8"))
ev = pd.read_parquet(os.path.join(PRESS_DIR, "pressure_events_player.parquet"))
sq = pd.read_parquet(os.path.join(PRESS_DIR, "press_sequences.parquet"))
pfr = pd.read_parquet(os.path.join(PRESS_DIR, "pressure_frames.parquet"))
FPS, L, W, N_FRAMES = meta["fps"], meta["pitch_length"], meta["pitch_width"], meta["n_frames"]
TEAM_NAMES = {int(k): v for k, v in meta["team_names"].items()}
ATTACK_DIR = {int(k): v for k, v in meta["attack_dir"].items()}
PEOPLE = {int(k): v for k, v in meta["people"].items()}
NUM = {t: v["number"] for t, v in PEOPLE.items()}
LABEL = {t: f"{v['team_name'] or ''} {v['number'] if v['number'] is not None else ''} {v['name']}".strip() for t, v in PEOPLE.items()}
sq["pressers"] = sq.pressers.astype(str).map(lambda s: [int(x) for x in s.split(",") if x])
print(f"press sequences {len(sq)} | player presses {len(ev)} | teams {TEAM_NAMES} | attack dir {ATTACK_DIR}")


def along_x(x, team):          # pitch x seen from `team` attacking left -> right (0 = own goal, 105 = opponent goal)
    return x if ATTACK_DIR[team] == 1 else L - x


def along_y(y, team):          # pitch y seen from `team`: 0 = its LEFT touchline, 68 = its right
    return y if ATTACK_DIR[team] == 1 else W - y


def third_of(x): return "own" if x < 35 else ("middle" if x < 70 else "attacking")
def lane_of(y): return "left" if y < W / 3 else ("right" if y > 2 * W / 3 else "centre")


live = pf[~pf.is_stoppage & pf.pitch_x.notna()].copy()
live["y_att"] = np.where(live.attack_dir.astype(float) == 1, live.pitch_y, W - live.pitch_y)

## 1. Inferred positions (no names): average position in possession -> line (DEF/MID/FWD) and side
Lines come from the gaps between the players' average depths (`LINE_GAP_M`), so a 4-3-3 and a 4-4-2 are found
as they are. The same lines give the **formation in possession**.

In [ ]:
def split_lines(depths, gap=LINE_GAP_M):
    """Players sorted by depth; a new line starts at the biggest gaps: the first two need > gap, a 4th line only
    if its gap is clearly bigger (> 1.6 x gap), so full-backs a few metres higher than the centre-backs stay one line."""
    order = np.argsort(depths); d = np.asarray(depths)[order]
    gaps = np.diff(d)
    cut = []
    for rank, c in enumerate(np.argsort(gaps)[::-1][:3]):
        if gaps[c] > (gap if rank < 2 else 1.6 * gap):
            cut.append(int(c))
    if not cut and len(gaps):
        cut = [int(np.argmax(gaps))]
    lines, start = [], 0
    for c in sorted(cut):
        lines.append(list(order[start:c + 1])); start = c + 1
    lines.append(list(order[start:]))
    return lines                                   # deepest line first


POS = {}
formation_in_poss = {}
for k, nm in TEAM_NAMES.items():
    inp = live[(live.team == k) & (live.possession_team == k)]
    g = inp.groupby("track_id").agg(role=("role", "first"), x=("x_att", "mean"), y=("y_att", "mean"), n=("frame_idx", "size"))
    g = g[g.n >= 5 * FPS]
    out = g[g.role == "player"]
    lines = split_lines(out.x.values)
    formation_in_poss[nm] = "-".join(str(len(l)) for l in lines)
    names = ["DEF", "MID", "FWD"] if len(lines) == 3 else (["DEF", "DM", "AM", "FWD"] if len(lines) == 4 else
                                                         ["DEF", "FWD"] if len(lines) == 2 else ["DEF"] * len(lines))
    for li, l in enumerate(lines):
        for i in l:
            tid = out.index[i]
            POS[tid] = {"line": names[li], "side": lane_of(out.y.iloc[i]), "x": out.x.iloc[i], "y": out.y.iloc[i]}
    for tid in g[g.role == "goalkeeper"].index:
        POS[tid] = {"line": "GK", "side": "centre", "x": g.loc[tid, "x"], "y": g.loc[tid, "y"]}
    print(f"{nm}: formation in possession {formation_in_poss[nm]}")
    for tid in g.sort_values("x").index:
        if tid in POS:
            print(f"   {LABEL[tid]:<22} {POS[tid]['line']:<4} {POS[tid]['side']:<7} avg depth {POS[tid]['x']:5.1f} m  "
                  f"avg width {POS[tid]['y']:5.1f} m")


def line_of(tid):
    return POS.get(int(tid), {}).get("line", "?") if pd.notna(tid) else "?"

## 2. Opponent possessions and how they started (goal kick / goalkeeper / restart / open play)

In [ ]:
poss = bt.possession_team.astype("float").reindex(range(N_FRAMES)).values
carr = bt.carrier_track_id.astype("float").reindex(range(N_FRAMES)).values
stop = bt.is_stoppage.reindex(range(N_FRAMES)).fillna(False).values.astype(bool)
pos_idx = pf.set_index(["frame_idx", "track_id"])[["pitch_x", "pitch_y", "x_att"]]


def ppos(f, t):
    try:
        r = pos_idx.loc[(int(f), int(t))]
        return (float(r.pitch_x), float(r.pitch_y)) if np.isfinite(r.pitch_x) else None
    except KeyError:
        return None


runs, cur = [], None
for f in range(N_FRAMES):
    p = poss[f]
    if np.isnan(p) or stop[f]:
        if cur: runs.append(cur); cur = None
        continue
    if cur and cur["team"] == int(p):
        cur["end"] = f
    else:
        if cur: runs.append(cur)
        cur = {"team": int(p), "start": f, "end": f}
if cur: runs.append(cur)
R = pd.DataFrame(runs)
first_c, dead_before, start_kind = [], [], []
for r in R.itertuples():
    f = r.start
    while f <= r.end and np.isnan(carr[f]): f += 1
    c = int(carr[f]) if f <= r.end else None
    b = r.start - 1
    while b >= 0 and np.isnan(carr[b]): b -= 1
    gap_s = (r.start - b - 1) / FPS
    first_c.append(c); dead_before.append(gap_s)
    pos0 = ppos(f, c) if c is not None else None
    x_own = along_x(pos0[0], r.team) if pos0 else np.nan
    if c is not None and line_of(c) == "GK" and gap_s >= GK_RESTART_GAP_S and x_own <= 16.5:
        start_kind.append("goal_kick")
    elif c is not None and line_of(c) == "GK":
        start_kind.append("gk_open_play")
    elif gap_s >= GK_RESTART_GAP_S:
        start_kind.append("restart")
    else:
        start_kind.append("open_play")
R["first_carrier"], R["dead_before_s"], R["start_kind"] = first_c, dead_before, start_kind
R["team_name"] = R.team.map(TEAM_NAMES)
R["duration_s"] = (R.end - R.start + 1) / FPS
print(pd.crosstab(R.team_name, R.start_kind, margins=True).to_string())

## 3. Every press sequence: where, on whom, in which situation
The location is the **carrier's** position at the start of the press, seen from the pressing team.

In [ ]:
run_start = R.start.values
rows = []
for s in sq.itertuples():
    k, opp = int(s.team), int(s.carrier_team)
    c = int(s.first_carrier)
    pc = ppos(s.start, c)
    i = np.searchsorted(run_start, s.start, side="right") - 1
    run = R.iloc[i] if i >= 0 and R.iloc[i].team == opp and R.iloc[i].end >= s.start else None
    since = (s.start - run.start) / FPS if run is not None else np.nan
    kind = run.start_kind if run is not None else "?"
    cl = line_of(c)
    if s.counter_press:
        sit = "counter_press"
    elif cl == "GK":
        sit = "on_goalkeeper"
    elif kind == "goal_kick" and since <= GOAL_KICK_PHASE_S:
        sit = "on_goal_kick"
    else:
        sit = {"DEF": "on_defenders", "MID": "on_midfielders", "DM": "on_midfielders", "AM": "on_midfielders",
               "FWD": "on_forwards"}.get(cl, "other")
    x = along_x(pc[0], k) if pc else np.nan
    y = along_y(pc[1], k) if pc else np.nan
    rows.append({"sequence_id": s.Index, "situation": sit, "carrier_line": cl,
                 "carrier_side": POS.get(c, {}).get("side", "?"), "possession_start": kind,
                 "s_since_possession_start": since, "ball_x": x, "ball_y": y,
                 "third": third_of(x) if np.isfinite(x) else None, "lane": lane_of(y) if np.isfinite(y) else None})
S = sq.join(pd.DataFrame(rows).set_index("sequence_id"))
S["first_carrier_label"] = S.first_carrier.map(LABEL)
S["presser_labels"] = S.pressers.map(lambda ps: ", ".join(LABEL[p] for p in ps))

SIT_ORDER = ["on_goalkeeper", "on_goal_kick", "on_defenders", "on_midfielders", "on_forwards", "counter_press", "other"]
sit_tab = S.groupby(["team_name", "situation"]).agg(presses=("start", "size"), regain_rate=("regain", "mean"),
                                                   mean_pressers=("n_pressers", "mean"), mean_duration_s=("duration_s", "mean"))
print(sit_tab.round(2).to_string())
react = S[S.s_since_possession_start.notna()].groupby("team_name").s_since_possession_start.median()
print("\nmedian time from the opponent getting the ball to the first press (s):", react.round(1).to_dict())

## 4. Where they press -- 3 x 3 grid (pressing team attacking left -> right)

In [ ]:
TH, LN = ["own", "middle", "attacking"], ["left", "centre", "right"]
GRID = {}
fig, axs = plt.subplots(1, len(TEAM_NAMES), figsize=(7 * len(TEAM_NAMES), 5))
for ax, (k, nm) in zip(np.atleast_1d(axs), TEAM_NAMES.items()):
    d = S[(S.team == k) & S.third.notna()]
    cnt = d.groupby(["lane", "third"]).size().unstack(fill_value=0).reindex(index=LN, columns=TH, fill_value=0)
    rr = d.groupby(["lane", "third"]).regain.mean().unstack().reindex(index=LN, columns=TH)
    GRID[nm] = {"count": cnt.to_dict(), "regain_rate": rr.round(3).to_dict()}
    tot = max(cnt.values.sum(), 1)
    ax.add_patch(Rectangle((0, 0), L, W, fill=False, lw=2))
    for xi, th in enumerate(TH):
        for yi, ln in enumerate(LN):
            n = cnt.loc[ln, th]
            ax.add_patch(Rectangle((xi * 35, yi * W / 3), 35, W / 3, color="tab:red", alpha=0.08 + 0.8 * n / cnt.values.max()
                                   if cnt.values.max() else 0.05))
            ax.text(xi * 35 + 17.5, yi * W / 3 + W / 6, f"{n}  ({n / tot:.0%})\nregain {rr.loc[ln, th]:.0%}"
                    if n else "0", ha="center", va="center", fontsize=10)
    ax.set_xlim(-2, L + 2); ax.set_ylim(W + 2, -2); ax.set_aspect("equal")
    ax.set_title(f"{nm} press sequences ({len(d)}) -- attacking ->\nrows: left / centre / right (from {nm})")
    ax.set_xticks([17.5, 52.5, 87.5]); ax.set_xticklabels(["own third", "middle third", "attacking third"]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "press_grid.png"), dpi=110); plt.show()

## 5. Marking: man-oriented (1 v 1) or zonal?
In frames where the team presses (opponent has the ball in its own half), each outfield defender's nearest opponent
within `MARK_RADIUS_M` is his "mark". A player is **man-oriented** when he keeps the same mark and moves *with him*
(his sideways position follows the opponent more than the ball); **zonal** when his marks change and he follows the ball.
This is an indicator from movement, not a ground truth.

In [ ]:
MARK = {}
pairs_all = []
for k, nm in TEAM_NAMES.items():
    opp = 1 - k
    fr = live[(live.possession_team == opp) & (live.frame_idx % MARK_SAMPLE_EVERY == 0)]
    ball_x = bt.ball_pitch_x.reindex(fr.frame_idx.unique())
    own_half = set(ball_x[[along_x(x, opp) < 52.5 if np.isfinite(x) else False for x in ball_x.values]].index)
    fr = fr[fr.frame_idx.isin(own_half) & (fr.role == "player")]
    D, O = fr[fr.team == k], fr[fr.team == opp]
    m = D.merge(O[["frame_idx", "track_id", "pitch_x", "pitch_y"]], on="frame_idx", suffixes=("", "_o"))
    m["d"] = np.hypot(m.pitch_x - m.pitch_x_o, m.pitch_y - m.pitch_y_o)
    m = m.loc[m.groupby(["frame_idx", "track_id"]).d.idxmin()]
    m = m[m.d <= MARK_RADIUS_M]
    by = bt.ball_pitch_y
    rows = []
    for tid, g in m.groupby("track_id"):
        if len(g) < 20:
            continue
        top, n_top = Counter(g.track_id_o).most_common(1)[0]
        share = n_top / len(g)
        gg = g[g.track_id_o == top].sort_values("frame_idx")
        c_opp = np.corrcoef(gg.pitch_y, gg.pitch_y_o)[0, 1] if len(gg) > 10 else np.nan
        c_ball = np.corrcoef(gg.pitch_y, by.reindex(gg.frame_idx).values)[0, 1] if len(gg) > 10 else np.nan
        man = share >= 0.6 and np.isfinite(c_opp) and c_opp > (c_ball if np.isfinite(c_ball) else -1) + 0.1
        rows.append({"player": LABEL[tid], "pos": f"{POS.get(tid, {}).get('line', '?')}-{POS.get(tid, {}).get('side', '?')}",
                     "main_mark": LABEL[top], "share_on_main_mark": share, "mean_dist_m": gg.d.mean(),
                     "follows_mark_corr": c_opp, "follows_ball_corr": c_ball, "man_oriented": man, "frames": len(g)})
    T = pd.DataFrame(rows).sort_values("share_on_main_mark", ascending=False)
    MARK[nm] = T
    idx = T.man_oriented.mean() if len(T) else np.nan
    print(f"\n{nm}: man-orientation index {idx:.0%} of outfield players "
          f"({'mostly man-oriented / 1 v 1' if idx >= 0.5 else 'mostly zonal'}) in the press")
    print(T.round(2).to_string(index=False))
    pairs_all += [dict(team=nm, **r) for r in rows if r["man_oriented"]]

## 6. Pressing shape: average out-of-possession shape in high / mid / low block
Block height = where the ball is, seen from the defending team (attacking third = high press, middle = mid block,
own third = low block). Positions are averaged only over frames where at least `SHAPE_MIN_VISIBLE` outfield players
of the team are on screen.

In [ ]:
SHAPES = {}
fig, axs = plt.subplots(len(TEAM_NAMES), 3, figsize=(18, 5.5 * len(TEAM_NAMES)))
bx = bt.ball_pitch_x
for row_i, (k, nm) in enumerate(TEAM_NAMES.items()):
    opp = 1 - k
    d = live[(live.team == k) & (live.role == "player") & (live.possession_team == opp)].copy()
    vis = d.groupby("frame_idx").size()
    d = d[d.frame_idx.isin(vis[vis >= SHAPE_MIN_VISIBLE].index)]
    d["ball_x_att"] = along_x(bx.reindex(d.frame_idx).values, k)
    d["block"] = np.select([d.ball_x_att >= 70, d.ball_x_att >= 35], ["high", "mid"], "low")
    d.loc[d.ball_x_att.isna(), "block"] = None
    SHAPES[nm] = {}
    for ci, blk in enumerate(["high", "mid", "low"]):
        ax = axs[row_i, ci] if len(TEAM_NAMES) > 1 else axs[ci]
        b = d[d.block == blk]
        ax.add_patch(Rectangle((0, 0), L, W, fill=False, lw=1.5)); ax.axvline(52.5, color="grey", lw=0.8)
        for xx in (35, 70): ax.axvline(xx, color="grey", lw=0.5, ls=":")
        if b.frame_idx.nunique() < 2 * FPS:
            ax.set_title(f"{nm} {blk} block: not enough frames"); ax.set_xlim(-2, L + 2); ax.set_ylim(W + 2, -2)
            ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); continue
        avg = b.groupby("track_id").agg(x=("x_att", "mean"), y=("y_att", "mean"), n=("frame_idx", "size"))
        avg = avg[avg.n >= max(10, 0.2 * b.frame_idx.nunique())]
        lines = split_lines(avg.x.values)
        form = "-".join(str(len(l)) for l in lines)
        per = b.groupby("frame_idx").agg(xmin=("x_att", "min"), xmax=("x_att", "max"), ymin=("y_att", "min"), ymax=("y_att", "max"))
        width, depth = float((per.ymax - per.ymin).median()), float((per.xmax - per.xmin).median())
        line_x = [float(avg.x.iloc[l].mean()) for l in lines]
        SHAPES[nm][blk] = {"formation": form, "frames": int(b.frame_idx.nunique()), "width_m": round(width, 1),
                           "depth_m": round(depth, 1), "last_line_m_from_goal": round(line_x[0], 1),
                           "line_heights_m": [round(v, 1) for v in line_x],
                           "gaps_between_lines_m": [round(b_ - a_, 1) for a_, b_ in zip(line_x, line_x[1:])],
                           "players": {LABEL[t]: [round(r.x, 1), round(r.y, 1)] for t, r in avg.iterrows()}}
        col = np.array(TEAM_COLORS.get(nm, (120, 120, 120))[::-1]) / 255
        from scipy.spatial import ConvexHull
        if len(avg) >= 3:
            h = ConvexHull(avg[["x", "y"]].values)
            ax.add_patch(Polygon(avg[["x", "y"]].values[h.vertices], color=col, alpha=0.15))
        for l in lines:                                                  # connect each line left -> right
            ln_ = avg.iloc[l].sort_values("y")
            ax.plot(ln_.x, ln_.y, color=col, lw=1, alpha=0.5)
        ax.scatter(avg.x, avg.y, s=380, color=col, edgecolor="white", zorder=3)
        for t, r in avg.iterrows():
            ax.text(r.x, r.y, str(NUM.get(t, "")), color="white", ha="center", va="center", fontsize=9, weight="bold", zorder=4)
        ax.set_xlim(-2, L + 2); ax.set_ylim(W + 2, -2); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{nm} -- {blk} block: {form}   (width {width:.0f} m, depth {depth:.0f} m, {b.frame_idx.nunique() / FPS:.0f} s)")
plt.suptitle("Out-of-possession shape (each team attacking left -> right; numbers = shirt numbers)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "press_shapes.png"), dpi=110); plt.show()
for nm, s in SHAPES.items():
    for blk, v in s.items():
        print(f"{nm} {blk:>4}: {v['formation']:<8} last line {v['last_line_m_from_goal']} m | lines {v['line_heights_m']} | "
              f"width {v['width_m']} m depth {v['depth_m']} m")

## 7. Real clips, chosen automatically
For each team x situation the best examples (most pressers, longest, regain first), spaced in time. Each clip: the
match video with the pressing team's shape (filled hull), pressers -> carrier lines, a 5 m ring on the carrier (drawn
on the pitch through the homography), every player's shirt number, a banner, and a mini-pitch with both teams.

In [ ]:
def pick_clips(S, n=CLIPS_PER_SITUATION):
    picks = []
    for (nm, sit), g in S.groupby(["team_name", "situation"]):
        g = g.assign(score=g.n_pressers * 2 + g.regain * 3 + g.duration_s.clip(upper=4)).sort_values("score", ascending=False)
        chosen = []
        for r in g.itertuples():
            if all(abs(r.start - c.start) > 10 * FPS for c in chosen):
                chosen.append(r)
            if len(chosen) >= n:
                break
        picks += [(nm, sit, i + 1, c) for i, c in enumerate(chosen)]
    return picks


PICKS = pick_clips(S)
print(f"{len(PICKS)} clips selected:")
for nm, sit, i, r in PICKS:
    print(f"  {nm} {sit:<15} #{i}: {int(r.start / FPS // 60):02d}:{r.start / FPS % 60:04.1f}  pressers {r.n_pressers} "
          f"outcome {r.outcome}  carrier {r.first_carrier_label}")


def _load_pickle(p):
    class _C(pickle.Unpickler):
        def find_class(self, m, n):
            return super().find_class(m.replace("numpy._core", "numpy.core", 1) if m.startswith("numpy._core") else m, n)
    b = open(p, "rb").read()
    try:
        return pickle.loads(b)
    except ModuleNotFoundError:
        return _C(io.BytesIO(b)).load()


def render_clip(nm, sit, i, r, HC, cap, fps_v):
    k = [t for t, v in TEAM_NAMES.items() if v == nm][0]
    s0, s1 = max(0, int(r.start - CLIP_PRE_S * FPS)), min(N_FRAMES - 1, int(r.end + CLIP_POST_S * FPS))
    rows = pf[(pf.frame_idx >= s0) & (pf.frame_idx <= s1)].groupby("frame_idx")
    prs = pfr[(pfr.frame_idx >= s0) & (pfr.frame_idx <= s1) & (pfr.team == k)].groupby("frame_idx")
    rows_d = {f: g for f, g in rows}; prs_d = {f: set(g.track_id) for f, g in prs}
    cap.set(cv2.CAP_PROP_POS_FRAMES, s0)
    Wv, Hv = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    sc = Hv / 1080
    path = os.path.join(CLIP_DIR, f"{nm}_{sit}_{i}_{int(r.start)}.mp4")
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps_v, (Wv, Hv))
    pcol, ocol = TEAM_COLORS.get(nm, (0, 0, 255)), (200, 200, 200)
    banner = (f"{nm} | {sit.replace('_', ' ')} | {r.third or '?'} third, {r.lane or '?'} | pressers: {r.n_pressers} "
              f"| outcome: {r.outcome.replace('_', ' ')}")
    mw, mh = int(315 * sc), int(204 * sc); msc = mw / L
    for f in range(s0, s1 + 1):
        ok, img = cap.read()
        if not ok:
            break
        g = rows_d.get(f)
        H = HC[f] if isinstance(HC, list) and f < len(HC) else None
        Hinv = np.linalg.inv(np.asarray(H, np.float64)) if H is not None else None
        car = int(carr[f]) if not np.isnan(carr[f]) else None
        if g is not None:
            mine = g[(g.team == k).fillna(False) & (g.role == "player")]
            if len(mine) >= 3:                                         # pressing team's shape
                pts = np.stack([((mine.x1 + mine.x2) / 2).values, mine.y2.values], 1).astype(np.int32)
                hull = cv2.convexHull(pts); ov = img.copy()
                cv2.fillPoly(ov, [hull], pcol); cv2.addWeighted(ov, 0.18, img, 0.82, 0, img)
                cv2.polylines(img, [hull], True, pcol, max(1, int(2 * sc)), cv2.LINE_AA)
            crow = g[g.track_id == car]
            if len(crow) and Hinv is not None and np.isfinite(crow.pitch_x.iloc[0]):   # 5 m ring on the pitch
                c = crow.iloc[0]; ang = np.linspace(0, 2 * np.pi, 48)
                circ = np.stack([c.pitch_x + 5 * np.cos(ang), c.pitch_y + 5 * np.sin(ang)], 1) * PX_PER_METER
                ip = cv2.perspectiveTransform(circ[None].astype(np.float32), Hinv).reshape(-1, 2).astype(np.int32)
                cv2.polylines(img, [ip], True, (0, 255, 255), max(1, int(2 * sc)), cv2.LINE_AA)
            for t in g.itertuples():
                fx, fy = int((t.x1 + t.x2) / 2), int(t.y2)
                is_p = t.track_id in prs_d.get(f, set())
                mine_t = (not pd.isna(t.team)) and int(t.team) == k
                col = pcol if mine_t else (ocol if t.role != "referee" else (30, 30, 30))
                rx = int(max((t.x2 - t.x1) * 0.6, 8 * sc))
                cv2.ellipse(img, (fx, fy), (rx, max(3, rx // 3)), 0, 0, 360, col, int((4 if is_p else 2) * sc) or 1, cv2.LINE_AA)
                if is_p and len(crow):
                    cx_, cy_ = int((crow.x1.iloc[0] + crow.x2.iloc[0]) / 2), int(crow.y2.iloc[0])
                    cv2.line(img, (fx, fy), (cx_, cy_), (0, 255, 255), max(1, int(2 * sc)), cv2.LINE_AA)
                if NUM.get(t.track_id) is not None:
                    txt = str(NUM[t.track_id]); (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.5 * sc, 1)
                    cv2.rectangle(img, (fx - tw // 2 - 3, int(t.y1) - th - 9), (fx + tw // 2 + 3, int(t.y1) - 3), col, -1)
                    cv2.putText(img, txt, (fx - tw // 2, int(t.y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5 * sc, (255, 255, 255),
                                max(1, int(sc)), cv2.LINE_AA)
                if t.track_id == car:
                    s_ = int(8 * sc); tip = int(t.y1) - int(14 * sc)
                    cv2.fillPoly(img, [np.array([[fx, tip], [fx - s_, tip - int(1.5 * s_)], [fx + s_, tip - int(1.5 * s_)]], np.int32)],
                                 (0, 165, 255), cv2.LINE_AA)
            # mini pitch, the pressing team attacking left -> right
            x0, y0 = Wv - mw - int(15 * sc), Hv - mh - int(15 * sc)
            ov = img.copy(); cv2.rectangle(ov, (x0, y0), (x0 + mw, y0 + mh), (40, 110, 40), -1)
            cv2.addWeighted(ov, 0.85, img, 0.15, 0, img)
            cv2.rectangle(img, (x0, y0), (x0 + mw, y0 + mh), (235, 235, 235), 1)
            cv2.line(img, (x0 + mw // 2, y0), (x0 + mw // 2, y0 + mh), (235, 235, 235), 1)
            mp = []
            for t in g.itertuples():
                if not np.isfinite(t.pitch_x) or t.role == "referee":
                    continue
                px, py = along_x(t.pitch_x, k), along_y(t.pitch_y, k)
                q = (int(x0 + px * msc), int(y0 + py * msc))
                mine_t = (not pd.isna(t.team)) and int(t.team) == k
                if mine_t and t.role == "player":
                    mp.append(q)
                cv2.circle(img, q, max(3, int(5 * sc)), pcol if mine_t else ocol, -1, cv2.LINE_AA)
            if len(mp) >= 3:
                cv2.polylines(img, [cv2.convexHull(np.array(mp, np.int32))], True, pcol, 1, cv2.LINE_AA)
            bxy = (bt.ball_pitch_x.get(f, np.nan), bt.ball_pitch_y.get(f, np.nan))
            if np.isfinite(bxy[0]):
                cv2.circle(img, (int(x0 + along_x(bxy[0], k) * msc), int(y0 + along_y(bxy[1], k) * msc)),
                           max(2, int(3 * sc)), (0, 255, 255), -1, cv2.LINE_AA)
        ov = img.copy(); cv2.rectangle(ov, (0, 0), (Wv, int(40 * sc)), (20, 20, 20), -1); cv2.addWeighted(ov, 0.7, img, 0.3, 0, img)
        live_tag = "  << PRESS" if r.start <= f <= r.end else ""
        cv2.putText(img, f"{int(f / FPS // 60):02d}:{f / FPS % 60:04.1f}  {banner}{live_tag}", (int(12 * sc), int(27 * sc)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65 * sc, (255, 255, 255), max(1, int(sc)), cv2.LINE_AA)
        out.write(img)
    out.release()
    return path


CLIP_INDEX = []
if RENDER_CLIPS and PICKS:
    HC = _load_pickle(HOMOGRAPHY_PATH)
    cap = cv2.VideoCapture(VIDEO_PATH); assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
    fps_v = cap.get(cv2.CAP_PROP_FPS) or FPS
    for nm, sit, i, r in PICKS:
        p_ = render_clip(nm, sit, i, r, HC, cap, fps_v)
        CLIP_INDEX.append({"team": nm, "situation": sit, "rank": i, "start_frame": int(r.start), "end_frame": int(r.end),
                           "time": f"{int(r.start / FPS // 60):02d}:{r.start / FPS % 60:04.1f}", "pressers": r.presser_labels,
                           "carrier": r.first_carrier_label, "outcome": r.outcome, "third": r.third, "lane": r.lane,
                           "path": p_})
        print("  saved", p_)
    cap.release()
pd.DataFrame(CLIP_INDEX).to_csv(os.path.join(CLIP_DIR, "clip_index.csv"), index=False, encoding="utf-8-sig")

## 8. Team pressing report (JSON, input of the LLM step) + save tables

In [ ]:
report = {}
for k, nm in TEAM_NAMES.items():
    d = S[S.team == k]
    T = MARK.get(nm, pd.DataFrame())
    report[nm] = {
        "formation_in_possession": formation_in_poss.get(nm),
        "press_sequences": int(len(d)),
        "regain_rate": round(float(d.regain.mean()), 3) if len(d) else None,
        "where_grid": GRID.get(nm),
        "third_share": d.third.value_counts(normalize=True).round(3).to_dict(),
        "lane_share": d.lane.value_counts(normalize=True).round(3).to_dict(),
        "situations": {sit: {"presses": int(len(g)), "regain_rate": round(float(g.regain.mean()), 3),
                             "mean_pressers": round(float(g.n_pressers.mean()), 2)} for sit, g in d.groupby("situation")},
        "median_reaction_s": round(float(d.s_since_possession_start.median()), 2) if d.s_since_possession_start.notna().any() else None,
        "marking": {"man_orientation_index": round(float(T.man_oriented.mean()), 3) if len(T) else None,
                    "man_oriented_pairs": [{"player": r["player"], "marks": r["main_mark"],
                                            "share": round(r["share_on_main_mark"], 2)} for _, r in T[T.man_oriented].iterrows()] if len(T) else []},
        "shape_by_block": SHAPES.get(nm, {}),
        "clips": [c for c in CLIP_INDEX if c["team"] == nm],
    }
with open(os.path.join(OUT_DIR, "team_pressing_report.json"), "w", encoding="utf-8") as fh:
    json.dump(report, fh, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))
S2 = S.copy(); S2["pressers"] = S2.pressers.map(lambda v: ",".join(map(str, v)))
S2.to_parquet(os.path.join(OUT_DIR, "press_sequences_situations.parquet"))
R.to_parquet(os.path.join(OUT_DIR, "possessions.parquet"))
pd.concat([T.assign(team=nm) for nm, T in MARK.items()]).to_csv(os.path.join(OUT_DIR, "marking.csv"), index=False, encoding="utf-8-sig")
print(f"Saved to {OUT_DIR}: team_pressing_report.json, press_sequences_situations, possessions, marking.csv, "
      f"press_grid.png, press_shapes.png, clips/ ({len(CLIP_INDEX)})")
for nm, r in report.items():
    print(f"\n{nm}: {r['press_sequences']} presses, regain {r['regain_rate']} | thirds {r['third_share']} | lanes {r['lane_share']}")
    print("   situations:", {k_: v["presses"] for k_, v in r["situations"].items()})
    print("   shapes:", {b: v["formation"] for b, v in r["shape_by_block"].items()},
          "| man-orientation:", r["marking"]["man_orientation_index"])